<a href="https://colab.research.google.com/github/kushalpatel92/cs521/blob/main/hw3/q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/structuredllm/syncode.git
import torch
from syncode import SyncodeLogitsProcessor
from syncode import Grammar
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda'
model_name = "microsoft/phi-2"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

  Cloning https://github.com/structuredllm/syncode.git to /tmp/pip-req-build-162ccqgy
  Running command git clone --filter=blob:none --quiet https://github.com/structuredllm/syncode.git /tmp/pip-req-build-162ccqgy
  Resolved https://github.com/structuredllm/syncode.git to commit 4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.7 MB/s eta 0:00:00
  Created wheel for syncode: filename=syncode-0.4.16-py3-none-any.w

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [11]:
grammar_str = r"""
start: NUMBER " " OP " " NUMBER "\n"
NUMBER: /\(-[0-9]{1,10}([.][0-9]{0,6})?\)|[0-9]{1,10}([.][0-9]{0,6})?/
OP: "+" | "-" | "*" | "/"
"""
arith_grammar = Grammar(grammar_str)
syncode_logits_processor = SyncodeLogitsProcessor(grammar=arith_grammar, tokenizer=tokenizer, parse_output_only=True)

FEW_SHOT = """Question: What is 27.65 plus 19.7?
Expression: 27.65 + 19.7

Question: What is -372. minus 22.89?
Expression: (-372.) - 22.89

Question: What is 327.5 multiplied by -11?
Expression: 327.5 * (-11)

Question: What is 100 divided by 8.?
Expression: 100 / 8.

Question: What is -5 plus 5?
Expression: (-5) + 5

Question: Subtract 4.5 from 20.
Expression: 20 - 4.5

Question: What is the product of 6.2 and 3?
Expression: 6.2 * 3

"""

[2026-10-28 03:09:03,761-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_1152735111_50257.pkl exist: False
[2026-10-28 03:09:03,762-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_1152735111_50257.pkl.
[2026-10-28 03:09:04,267-syncode.mask_store.fsm_set] - 4 FSMs with 47 states initialized in 0.00 seconds
[2026-10-28 03:09:04,268-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-28 03:09:04,269-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 16 to 5


100%|██████████| 47/47 [00:08<00:00,  5.24it/s]


[2026-10-28 03:09:13,403-syncode.mask_store.mask_store] - Time taken to create mask store: 9.63 seconds


In [12]:
def run_syncode(textual_query):
    prompt = FEW_SHOT + "Question: " + textual_query.strip() + "\nExpression: "
    syncode_logits_processor.reset()

    inputs = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    output = model.generate(
        inputs,
        max_new_tokens=20,
        do_sample=False,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        logits_processor=[syncode_logits_processor],
    )
    text = tokenizer.decode(output[0][len(inputs[0]):], skip_special_tokens=True)

    print(repr(text))
    first, op, second = text.split()[:3]
    x, y = float(first.strip("()")), float(second.strip("()"))

    if op == "+":
        answer = x + y
    elif op == "-":
        answer = x - y
    elif op == "*":
        answer = x * y
    else:
        answer = x / y if y != 0 else float("nan")

    return ([x, y], op, round(float(answer), 6))

In [14]:
tests = [
    ("What is 327. multiplied by 11.0?", 3597.0),
    ("What is 45.1 plus 23.54?", 68.64),
    ("What is 120.4 divided by 4.0?", 30.1),
    ("What is -500 minus 123.25?", -623.25),
    ("Subtract 7 from 50.5", 43.5),
    ("What is 8 times 2.5?", 20.0),
]

for q, expected in tests:
    result = run_syncode(q)
    status = "OK" if abs(result[2] - expected) < 1e-6 else "WRONG"
    print(status, q, "->", result)

'327. * 11.0\n'
OK What is 327. multiplied by 11.0? -> ([327.0, 11.0], '*', 3597.0)
'45.1 + 23.54\n'
OK What is 45.1 plus 23.54? -> ([45.1, 23.54], '+', 68.64)
'120.4 / 4.0\n'
OK What is 120.4 divided by 4.0? -> ([120.4, 4.0], '/', 30.1)
'(-500) - 123.25\n'
OK What is -500 minus 123.25? -> ([-500.0, 123.25], '-', -623.25)
'50.5 - 7\n'
OK Subtract 7 from 50.5 -> ([50.5, 7.0], '-', 43.5)
'8 * 2.5\n'
OK What is 8 times 2.5? -> ([8.0, 2.5], '*', 20.0)
